<a href="https://colab.research.google.com/github/GAURAV4478/cookbook/blob/sql-agent-notebook/examples/langchain/SQL_Agent_Gemini_LangChain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##### Copyright 2026 Google LLC.

In [31]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini API: SQL Agent using LangChain and SQLite

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Template.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

<!-- Community Contributor Badge -->
<table>
  <tr>
    <!-- Author Avatar Cell -->
    <td bgcolor="#d7e6ff">
      <a href="https://github.com/GAURAV4478" target="_blank" title="View Gaurav's profile on GitHub">
        <img src="https://github.com/GAURAV4478.png?size=100"
             alt="GAURAV4478's GitHub avatar"
             width="100"
             height="100">
      </a>
    </td>
    <!-- Text Content Cell -->
    <td bgcolor="#d7e6ff">
      <h2><font color='black'>This notebook was contributed by <a href="https://github.com/GAURAV4478" target="_blank"><font color='#217bfe'><strong>Gaurav Thakur</strong></font></a>.</font></h2>
      <h5><font color='black'>
        <a href="https://linkedin.com/in/gauravthakur7" target="_blank"><font color="#078efb">LinkedIn</font></a><br>
        <a href="https://github.com/GAURAV4478" target="_blank"><font color="#078efb">GitHub</font></a>
      </h5></font><br>
  </tr>
</table>

This notebook demonstrates how to build an autonomous SQL Agent
using the Gemini API and LangChain. Unlike a traditional chain-based
approach, a SQL Agent can reason about your question, decide which
queries to run, and self-correct errors — all on its own.

By the end of this notebook, you will be able to:
- Set up a SQLite database with sample data
- Create a LangChain SQL Agent powered by Gemini
- Query the database using plain natural language

## Setup

### Install SDK

In [32]:
%pip install -U -q "google-genai>=2.9.0" langchain langchain-community langchain-google-genai

In [33]:
import sqlite3
import pandas as pd
from sklearn.datasets import fetch_openml

from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import create_sql_agent
from langchain_google_genai import ChatGoogleGenerativeAI
from IPython.display import Markdown

### Set up your API key

You'll need a Gemini API key to run this notebook.

1. Get your free API key from [Google AI Studio](https://aistudio.google.com/apikey)
2. In Colab, click the 🔑 **Secrets** icon in the left sidebar
3. Add a new secret with name `GEMINI_API_KEY` and paste your key as the value
4. Enable notebook access for the secret

In [34]:
from google.colab import userdata
from google import genai

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=GEMINI_API_KEY)

Select the model you want to use in this guide:

In [35]:
MODEL_ID = "gemini-3.1-flash-lite" # @param ["gemini-3.1-flash-lite", "gemini-2.5-flash", "gemini-3.5-flash", "gemini-2.5-pro", "gemini-3-flash-preview", "gemini-3.1-pro-preview"] {"allow-input":true, isTemplate: true}

# Ideally order the model by "cabability" ie. generation then within generation
# 8b/flash-lite then flash then pro

## Setting up the database

In this section, you will create a SQLite database using the Titanic dataset.
The dataset contains information about passengers including age, gender, ticket class, fare, and survival status.


1. To query a database, you first need to set one up. Here you will use the Titanic dataset loaded directly from OpenML.

In [36]:
titanic = fetch_openml(name="titanic", version=1, as_frame=True)
df = titanic.frame[["pclass", "survived", "name", "sex", "age", "fare", "embarked"]].copy()

2. **Clean the data:** Convert columns to correct data types and remove rows with missing values.

In [37]:
df["age"] = pd.to_numeric(df["age"], errors="coerce")
df["fare"] = pd.to_numeric(df["fare"], errors="coerce")
df["survived"] = df["survived"].astype(int)
df["pclass"] = df["pclass"].astype(int)
df.dropna(inplace=True)

3. **Create the SQLite database:** Store the cleaned data in a SQLite database called `titanic.db` inside a table named `passengers`.

In [38]:
conn = sqlite3.connect("titanic.db")
df.to_sql("passengers", conn, index=False, if_exists="replace")

1043

## Create the SQL agent

Now that the database is ready, create a SQL agent using LangChain and Gemini.

In [39]:
# Initialize Gemini as the LLM
llm = ChatGoogleGenerativeAI(
    model=MODEL_ID,
    google_api_key=GEMINI_API_KEY
)

Create a SQLDatabase object to connect to the Titanic database.

In [40]:
db = SQLDatabase.from_uri("sqlite:///titanic.db")
print(db.get_table_info())


CREATE TABLE passengers (
	pclass INTEGER, 
	survived INTEGER, 
	name TEXT, 
	sex TEXT, 
	age REAL, 
	fare REAL, 
	embarked TEXT
)

/*
3 rows from passengers table:
pclass	survived	name	sex	age	fare	embarked
1	1	Allen, Miss. Elisabeth Walton	female	29.0	211.3375	S
1	1	Allison, Master. Hudson Trevor	male	0.9167	151.55	S
1	0	Allison, Miss. Helen Loraine	female	2.0	151.55	S
*/


Create the SQL agent by combining the LLM and the database.

In [69]:
# Create the SQL agent
agent = create_sql_agent(
    llm=llm,
    db=db,
    verbose=False,
    agent_type="openai-tools",
    handle_parsing_errors=True,
    prefix="""You are a helpful data analyst assistant.
    When answering questions, always explain your findings in clear, simple English.
    Always provide the exact numbers from the database in your response."""
)

## Query the database

Now that the agent is ready, query the database using plain English. The agent will automatically generate the SQL query, execute it, and return the answer.

Before querying, define a helper function to display the question, the SQL query generated by the agent, and the final answer in a clean format.

In [70]:
# Helper function to run and display agent queries
def ask_agent(question):
    response = agent.invoke({"input": question})

    print(f"Question: {question}")
    print(f"Answer: {response['output'][0]['text']}")
    print("-" * 50)

Run the following queries to explore the Titanic dataset using natural language.

In [71]:
ask_agent("How many passengers survived and how many did not?")



> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{'tool_input': ''}`
responded: [{'type': 'text', 'text': '\n', 'index': 0}]

passengers
Invoking: `sql_db_schema` with `{'table_names': 'passengers'}`



CREATE TABLE passengers (
	pclass INTEGER, 
	survived INTEGER, 
	name TEXT, 
	sex TEXT, 
	age REAL, 
	fare REAL, 
	embarked TEXT
)

/*
3 rows from passengers table:
pclass	survived	name	sex	age	fare	embarked
1	1	Allen, Miss. Elisabeth Walton	female	29.0	211.3375	S
1	1	Allison, Master. Hudson Trevor	male	0.9167	151.55	S
1	0	Allison, Miss. Helen Loraine	female	2.0	151.55	S
*/
Invoking: `sql_db_query_checker` with `{'query': 'SELECT survived, COUNT(*) FROM passengers GROUP BY survived'}`


SELECT survived, COUNT(*) FROM passengers GROUP BY survived
Invoking: `sql_db_query` with `{'query': 'SELECT survived, COUNT(*) FROM passengers GROUP BY survived'}`


[(0, 618), (1, 425)][{'type': 'text', 'text': 'Out of the passengers in the database, 425 survived and 6